# Listwise reranker (open RankZephyr) over the ensemble's top-N

**Why (from the §11c gap decomposition).** The 0.61→0.957 gap is a *tight race*: buried eligibles get
a weak-positive judge score (+1.9 vs +5.8 for surfaced), the cross-encoders rate them *higher* than
surfaced docs but are too flat to override the LLM-dominated top-10. Every **pointwise** fix went null
(CoT §11a, judge-FT §11c — the FT judge made the tail *worse*). A **listwise** reranker is the one
paradigm mechanism-matched to a tight race: it reads the candidates *together* and can promote a +1.9
eligible over a false positive using cross-candidate context no pointwise score sees.

## Design
- **Open RankZephyr** (`castorini/rank_zephyr_7b_v1_full`) — purpose-built open listwise reranker.
  Zero-shot first (cheap test before any fine-tune); swap via `RANKER_MODEL`.
- **Honest base = out-of-fold TREC21** ensemble ranking (the CV that scored ~0.552, where eligibles
  are genuinely buried) — NOT the in-sample 0.65 that made CoT's bar unbeatable.
- **Sliding window** (w=20, s=10, own implementation) over the ensemble top-N; below N stays put.
  Sweep N so the window reaches buried eligibles.
- **Anti-gaming:** dev on TREC21 only; TREC22 touched once, later, only if this clears TREC21.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate lightgbm datasets pytrec_eval tqdm hf_transfer sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT       = '/content/drive/MyDrive/ct_data23'
# ROOT CAUSE (found via diagnostics): HF's new **Xet** transfer backend stalls on this instance,
# while the classic CDN streams fine at ~50MB/s. Disable Xet -> everything uses the working path.
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ.pop('HF_HUB_ENABLE_HF_TRANSFER', None)   # deprecated / ignored now
os.environ['HF_HOME'] = '/content/hf_cache'
EVAL_ROOT       = f'{DATA_ROOT}/evaluation'
TREC_ROOT       = f'{EVAL_ROOT}/trec_data'
KZ_ROOT         = f'{EVAL_ROOT}/kz_data'
FULLTEXT_CORPUS = f'{DATA_ROOT}/doc_texts_fulltext.txt'
FEATURE_CACHE   = f'{DATA_ROOT}/ensemble_features_full.npz'
META_CACHE      = f'{DATA_ROOT}/ensemble_features_full_meta.json'
LLM_SCORES      = f'{DATA_ROOT}/llm_reranker_scores.jsonl'
RANKER_MODEL    = 'castorini/rank_zephyr_7b_v1_full'   # fallback: 'Qwen/Qwen2.5-7B-Instruct'

DEV_SRC   = 'trec21'
RERANK_N  = [20]          # fair retry: one focused depth first (add 50 once the passages are verified)
WIN, STEP = 10, 5         # smaller window so eligibility-inclusive passages fit RankZephyr's 4096 ctx
# Corpus blob is ordered title->conditions->summary->detailed->interventions->ELIGIBILITY (last).
# 400-char head-only truncation NEVER reached eligibility (the rel2-vs-rel0 signal) -> confounded §11d.
# Fix: head (topicality) + tail (lands on eligibility). Verify with the inspection cell below.
PASSAGE_HEAD, PASSAGE_TAIL, QUERY_CHARS = 220, 1200, 500
# frozen CV-selected ensemble config (matches train_ensemble_full's best_cfg -> trec21 CV 0.552)
BEST_CFG = {'objective': 'lambdarank', 'metric': 'ndcg', 'ndcg_eval_at': [10],
            'learning_rate': 0.05, 'lambda_l2': 1.0, 'num_leaves': 15,
            'min_data_in_leaf': 20, 'verbose': -1}
ROUNDS = 50
os.environ['CTMATCH_DATA_ROOT'] = DATA_ROOT
print('config set | Xet disabled | passages = head', PASSAGE_HEAD, '+ tail', PASSAGE_TAIL, '| win', WIN)

In [ ]:
# ── DIAGNOSTIC: is the large-file transfer itself stalling on this instance? ─────────────────
# Streams ONE big RankZephyr shard from HF's CDN to LOCAL disk (not Drive) with a 30s READ timeout.
# If bytes stop arriving for 30s, requests raises -> we learn EXACTLY whether/where it dies. This
# isolates raw network egress from the HF downloader AND from Drive-FUSE write stalls.
import time, os, requests
from huggingface_hub import HfApi, hf_hub_url

REPO = 'castorini/rank_zephyr_7b_v1_full'
ENDPOINT = 'https://huggingface.co'          # change to 'https://hf-mirror.com' to test the mirror
shards = [s.rfilename for s in HfApi(endpoint=ENDPOINT).model_info(REPO).siblings
          if s.rfilename.endswith(('.safetensors', '.bin'))]
fn = shards[0]
url = hf_hub_url(REPO, fn, endpoint=ENDPOINT)
print('streaming', fn, 'from', ENDPOINT, '-> local disk, 30s read-timeout, stops at 2GB')

dst = '/content/_speedtest.bin'
t0 = last = time.time(); got = last_bytes = 0
try:
    with requests.get(url, stream=True, timeout=(30, 30)) as r:   # (connect, read) timeouts
        print('  HTTP', r.status_code, '| shard size %.1f GB' % (int(r.headers.get('content-length', 0)) / 1e9))
        with open(dst, 'wb') as f:
            for chunk in r.iter_content(1 << 20):                 # 1 MB chunks
                f.write(chunk); got += len(chunk); now = time.time()
                if now - last >= 3:
                    print('  %.2f GB   %.0f MB/s' % (got / 1e9, (got - last_bytes) / 1e6 / (now - last)))
                    last, last_bytes = now, got
                if got > 2e9:
                    print('  >>> 2 GB in %.0fs — NETWORK IS FINE. The stall is the HF downloader/Drive, not egress.'
                          % (time.time() - t0))
                    break
except requests.exceptions.RequestException as e:
    print('  >>> DIED at %.2f GB after %.0fs: %s' % (got / 1e9, time.time() - t0, type(e).__name__))
    print('  >>> a read-timeout = the connection went dead mid-transfer -> THIS INSTANCE network path is broken.')
    print('      Fix: factory-reset the runtime, or set ENDPOINT = https://hf-mirror.com above and re-run.')
finally:
    if os.path.exists(dst): os.remove(dst)

In [ ]:
# ── DIAGNOSTIC 2: does a DIFFERENT model download via the normal AutoModel path? ─────────────
# Pinpoints RankZephyr-specific vs general. Pulls a different org's ~3GB model through the SAME
# transformers/hf_hub download path RankZephyr uses.
#   completes  -> the problem is specific to the RankZephyr repo / its CDN objects.
#   hangs too  -> the HF download path (or network) is broken for everything large on this instance.
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

TEST_REPO = 'Qwen/Qwen2.5-1.5B-Instruct'   # ~3GB, different org/CDN objects than castorini
print('downloading', TEST_REPO, '...  (if this hangs >2-3 min like RankZephyr, the issue is general)')
t = time.time()
_tok = AutoTokenizer.from_pretrained(TEST_REPO)
_m = AutoModelForCausalLM.from_pretrained(TEST_REPO, dtype='auto')
print('  OK in %.0fs -> a DIFFERENT large model downloads fine -> the stall is RankZephyr-specific'
      % (time.time() - t))
del _m

In [ ]:
import json, numpy as np
from datasets import load_dataset
from ctmatch.evaluation.eval_utils import load_eval_datasets

# cached ensemble features + meta (base 8 features; llm rebuilt below exactly as cell-66128f71)
d = np.load(FEATURE_CACHE, allow_pickle=True)
Xtr, ytr, gtr = d['Xtr'], d['ytr'], [int(c) for c in d['gtr']]
with open(META_CACHE) as f:
    mtr = [tuple(x) for x in json.load(f)['mtr']]

llm_lookup = {}
with open(LLM_SCORES) as f:
    for line in f:
        r = json.loads(line); llm_lookup[(r['source'], r['topic_id'], r['doc_id'])] = r['llm_score']
FLOOR = min(llm_lookup.values()) - 5.0
raw = np.array([llm_lookup.get(m, np.nan) for m in mtr], dtype=np.float32)
scored = (~np.isnan(raw)).astype(np.float32)
val = np.where(np.isnan(raw), FLOOR, raw).astype(np.float32)
Xtr = np.column_stack([Xtr[:, :8], val, scored]).astype(np.float32)
FEATURES = ['bm25','bm25_rank','dense','dense_rank','rrf','clf_rel','clf_partial','v2_rel','llm_yesno','llm_scored']

# corpus text + topics
_idx = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
corpus_ids = [r['text'].strip() for r in _idx]
with open(FULLTEXT_CORPUS) as f:
    corpus_txt = [l.rstrip('\n') for l in f]
id2txt = dict(zip(corpus_ids, corpus_txt))
all_sets = load_eval_datasets(TREC_ROOT, KZ_ROOT)
topic2text = all_sets[DEV_SRC]['topic2text']
print(f'features {Xtr.shape} | topics loaded | corpus {len(id2txt):,}')

In [ ]:
import lightgbm as lgb

# out-of-fold ensemble predictions (same 5-fold seed as train_ensemble_full) -> honest TREC21 base
n_topics = len(gtr)
offsets = np.concatenate([[0], np.cumsum(gtr)]).astype(int)
topic_src = [mtr[offsets[t]][0] for t in range(n_topics)]
fold = np.random.default_rng(42).integers(0, 5, n_topics)

oof = np.zeros(len(ytr), dtype=np.float32)
for f in range(5):
    trn = [t for t in range(n_topics) if fold[t] != f]
    idx = np.concatenate([np.arange(offsets[t], offsets[t+1]) for t in trn]).astype(int)
    gt = [gtr[t] for t in trn]
    bst = lgb.train(BEST_CFG, lgb.Dataset(Xtr[idx], ytr[idx], group=gt, feature_name=FEATURES),
                    num_boost_round=ROUNDS)
    for t in (t for t in range(n_topics) if fold[t] == f):
        a, b = offsets[t], offsets[t+1]; oof[a:b] = bst.predict(Xtr[a:b])

def ndcg_at10(ordered_rels, pool_rels):
    g = [2.0**r - 1 for r in ordered_rels[:10]]
    dcg = sum(gg / np.log2(i + 2) for i, gg in enumerate(g))
    ideal = sorted(pool_rels, reverse=True)[:10]
    idcg = sum((2.0**r - 1) / np.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

# per TREC21 topic: candidate doc_ids ranked by the ensemble (base), + rel lookup
dev = []
for t in range(n_topics):
    if topic_src[t] != DEV_SRC: continue
    a, b = offsets[t], offsets[t+1]
    docs = [mtr[r][2] for r in range(a, b)]; rels = ytr[a:b].tolist(); sc = oof[a:b]
    order = np.argsort(-sc)
    ranked_docs = [docs[i] for i in order]
    rel_of = {docs[i]: int(rels[i]) for i in range(len(docs))}
    dev.append({'tid': mtr[a][1], 'ranked': ranked_docs, 'rel_of': rel_of, 'pool_rels': [int(r) for r in rels]})
base = np.mean([ndcg_at10([d['rel_of'][x] for x in d['ranked']], d['pool_rels']) for d in dev])
print(f'TREC21 out-of-fold ensemble base NDCG@10 = {base:.4f}  ({len(dev)} topics)  [target to beat]')

In [ ]:
# Robust model download via aria2c. Key vs HF's downloader: --lowest-speed-limit AUTO-ABORTS a stalled
# connection (<1MB/s) and retries, instead of hanging forever on a dead socket (your "never past 2%").
# 16 connections/file + resume + infinite retries. Writes to Drive so it persists across disconnects.
import os, subprocess
from huggingface_hub import HfApi, hf_hub_url

# If HF's CDN path from THIS Colab instance is the broken link, a mirror routes around it.
USE_MIRROR = True
if USE_MIRROR:
    os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'   # set False above to use the default HF CDN
ENDPOINT = os.environ.get('HF_ENDPOINT', 'https://huggingface.co')

get_ipython().system('apt-get -qq install -y aria2 > /dev/null 2>&1')
LOCAL_MODEL = f'{DATA_ROOT}/models/' + RANKER_MODEL.split('/')[-1]
os.makedirs(LOCAL_MODEL, exist_ok=True)
files = [s.rfilename for s in HfApi(endpoint=ENDPOINT).model_info(RANKER_MODEL).siblings]
print(f'{RANKER_MODEL}: {len(files)} files via {ENDPOINT} -> {LOCAL_MODEL}')

for fn in files:
    dst = os.path.join(LOCAL_MODEL, fn)
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print('have', fn); continue
    os.makedirs(os.path.dirname(dst) or '.', exist_ok=True)
    url = hf_hub_url(RANKER_MODEL, fn, endpoint=ENDPOINT)
    for attempt in range(20):
        try:
            subprocess.run(['aria2c', '-x16', '-s16', '-c', '--max-tries=0', '--retry-wait=5',
                            '--lowest-speed-limit=1M', '--timeout=60', '--connect-timeout=30',
                            '--console-log-level=warn', '--summary-interval=10',
                            '-d', os.path.dirname(dst) or '.', '-o', os.path.basename(dst), url],
                           check=True)
            break
        except subprocess.CalledProcessError:
            print(f'  retry {attempt+1} on {fn}')
print('\nmodel ready at', LOCAL_MODEL)

In [ ]:
import re, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Xet disabled in config -> from_pretrained downloads via the classic CDN (the path that works).
local = RANKER_MODEL
gpu = 'cuda'
tok = AutoTokenizer.from_pretrained(local)
model = AutoModelForCausalLM.from_pretrained(local, dtype=torch.bfloat16,
                                             attn_implementation='sdpa').to(gpu).eval()
SYS_RANK = 'You are RankLLM, an intelligent assistant that can rank passages based on their relevancy to the query.'

def _passage(d):
    # head = title/conditions (topicality); tail = the ELIGIBILITY section (last field in the blob).
    t = id2txt.get(d, '')
    if len(t) <= PASSAGE_HEAD + PASSAGE_TAIL:
        return t
    return t[:PASSAGE_HEAD] + ' … eligibility: ' + t[-PASSAGE_TAIL:]

def _window_prompt(query, passages):
    n = len(passages)
    s = (f'I will provide you with {n} passages, each indicated by a numerical identifier [].\n'
         f'Rank the passages based on their relevance to the search query: {query[:QUERY_CHARS]}.\n\n')
    for i, p in enumerate(passages, 1):
        s += f'[{i}] {p}\n'
    s += (f'\nSearch Query: {query[:QUERY_CHARS]}.\nRank the {n} passages above based on relevance to the '
          f'query, in descending order, using their identifiers. Format: [] > [] > ..., e.g. [2] > [1]. '
          f'Only respond with the ranking, do not say anything else.')
    return s

def _parse(text, n):
    seen = []
    for x in (int(m) for m in re.findall(r'\[(\d+)\]', text)):
        if 1 <= x <= n and x not in seen: seen.append(x)
    for x in range(1, n + 1):
        if x not in seen: seen.append(x)       # append any the model dropped (RankGPT convention)
    return [x - 1 for x in seen]

@torch.no_grad()
def _rank_window(query, passages):
    msgs = [{'role': 'system', 'content': SYS_RANK}, {'role': 'user', 'content': _window_prompt(query, passages)}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt',
                                  return_dict=True, truncation=True, max_length=4096).to(gpu)
    out = model.generate(**enc, max_new_tokens=len(passages) * 8 + 12, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    gen = out[0, enc['input_ids'].shape[1]:]
    return _parse(tok.decode(gen, skip_special_tokens=True), len(passages))

def listwise_rerank(query, doc_ids):
    passages = [_passage(d) for d in doc_ids]
    order = list(range(len(doc_ids))); end = len(doc_ids)
    while end > 0:
        start = max(0, end - WIN)
        win = order[start:end]
        perm = _rank_window(query, [passages[i] for i in win])
        order[start:end] = [win[j] for j in perm]
        if start == 0: break
        end -= STEP
    return [doc_ids[i] for i in order]

print('listwise reranker ready:', RANKER_MODEL)

In [ ]:
# VERIFY THE FIX before the full run: print the passages RankZephyr will actually receive for one
# topic's top docs. Confirm eligibility criteria (inclusion/exclusion) are now visible — that's the
# rel=2-vs-rel=0 signal the 400-char run was starving it of. If you don't see eligibility text here,
# adjust PASSAGE_TAIL up before running cell-run.
d0 = dev[0]
print('topic:', d0['tid'], '—', topic2text[d0['tid']][:200], '\n')
for doc in d0['ranked'][:2]:
    print('===', doc, '| true rel', d0['rel_of'][doc], '| passage sent to reranker ===')
    print(_passage(doc))
    print()

In [ ]:
from tqdm.auto import tqdm

print(f'base (ensemble, out-of-fold TREC21) NDCG@10 = {base:.4f}\n')
for N in RERANK_N:
    lw = []
    for d in tqdm(dev, desc=f'listwise N={N}'):
        head = d['ranked'][:N]
        new_head = listwise_rerank(topic2text[d['tid']], head)
        reordered = new_head + d['ranked'][N:]
        lw.append(ndcg_at10([d['rel_of'][x] for x in reordered], d['pool_rels']))
    lw = float(np.mean(lw))
    print(f'  N={N:<4d} listwise NDCG@10 = {lw:.4f}   delta {lw - base:+.4f}')
print('\nGate: a robust positive delta here (re-check across a couple of seeds/N) -> take it to the')
print('ensemble as a feature/rerank and, only then, the TREC22 one-shot. Flat/negative -> bank parity.')